<a href="https://colab.research.google.com/github/rah-ds/Cloud-Autoscaling-using-RL/blob/bmcgregor%2Fsimulator-refactor/notebooks/Experiment_DQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Experiment 1 - DQN

In [1]:
raw_url_utilities = 'https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Utils.py'
local_filename_utilities = 'DQN_Utils.py'
!wget -O {local_filename_utilities} {raw_url_utilities}

--2025-11-26 02:20:44--  https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9136 (8.9K) [text/plain]
Saving to: ‘DQN_Utils.py’

DQN_Utils.py        100%[===================>]   8.92K  --.-KB/s    in 0s      

2025-11-26 02:20:44 (67.9 MB/s) - ‘DQN_Utils.py’ saved [9136/9136]



In [2]:
raw_url_dqn_agent = 'https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Agent.py'
local_filename_dqn_agent = 'DQN_Agent.py'
!wget -O {local_filename_dqn_agent} {raw_url_dqn_agent}

--2025-11-26 02:20:45--  https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Agent.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2727 (2.7K) [text/plain]
Saving to: ‘DQN_Agent.py’

DQN_Agent.py        100%[===================>]   2.66K  --.-KB/s    in 0s      

2025-11-26 02:20:46 (28.7 MB/s) - ‘DQN_Agent.py’ saved [2727/2727]



In [3]:

raw_url = 'https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/autoscaling_env.py'



# Replace with the desired local filename for your script
local_filename = 'autoscaling_env.py'

!wget -O {local_filename} {raw_url}

--2025-11-26 02:20:47--  https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/autoscaling_env.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8002 (7.8K) [text/plain]
Saving to: ‘autoscaling_env.py’

autoscaling_env.py  100%[===================>]   7.81K  --.-KB/s    in 0s      

2025-11-26 02:20:48 (69.5 MB/s) - ‘autoscaling_env.py’ saved [8002/8002]



In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

df_usage = pd.read_csv('/content/drive/MyDrive/df_usage.csv')
display(df_usage.head())

,time_window,avg_cpu,avg_mem,active_machines
0,1970-01-01 00:05:00+00:00,0.006623,0.004912,9525
1,1970-01-01 00:06:00+00:00,0.003254,0.002733,3805
2,1970-01-01 00:07:00+00:00,0.003070,0.002770,4167
3,1970-01-01 00:08:00+00:00,0.001950,0.001823,4338
4,1970-01-01 00:09:00+00:00,0.001689,0.001468,5545


In [5]:
import sys
# Add the current directory to the system path to ensure Colab can find your local .py files
sys.path.append('.')


In [6]:
df_usage = df_usage.drop(columns=['active_machines']).copy()
display(df_usage.head())

,time_window,avg_cpu,avg_mem
0,1970-01-01 00:05:00+00:00,0.006623,0.004912
1,1970-01-01 00:06:00+00:00,0.003254,0.002733
2,1970-01-01 00:07:00+00:00,0.003070,0.002770
3,1970-01-01 00:08:00+00:00,0.001950,0.001823
4,1970-01-01 00:09:00+00:00,0.001689,0.001468


#DQN Experiment

In [7]:

from autoscaling_env import AutoScalingEnv
from DQN_Utils import train_agent, evaluate_agent, QNetwork, ReplayBuffer
from DQN_Agent import DQNAgent
import numpy as np
import sys
import matplotlib.pyplot as plt # Import missing plt for plotting
import torch # Import torch to define the device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1. Initialize the AutoScalingEnv environment for the DQN agent
env_dqn = AutoScalingEnv(df_usage)

# 2. Instantiate the DQNAgent
agent_dqn = DQNAgent(
    observation_space_shape=env_dqn.observation_space.shape,
    action_space_size=env_dqn.action_space.n,
    seed=0,
    device=device
)

# 3. Define the training hyperparameters for the DQN agent
num_episodes_dqn = 500
max_steps_per_episode_dqn = 1000
epsilon_start_dqn = 1.0
epsilon_end_dqn = 0.01
epsilon_decay_dqn = 0.995

# 4. Call the train_agent function
dqn_training_scores = train_agent(
    agent_dqn,
    env_dqn,
    num_episodes_dqn,
    max_steps_per_episode_dqn,
    epsilon_start_dqn,
    epsilon_end_dqn,
    epsilon_decay_dqn,
    split="train",
    use_random_windows = True
)

print("DQN Training Scores:", dqn_training_scores[:5]) # Display first 5 scores for brevity

# 5. Define the evaluation hyperparameters for the DQN agent
num_evaluation_episodes_dqn = 10
max_steps_per_evaluation_episode_dqn = 1000

# 6. Call the evaluate_agent functions


# Validation evaluation
dqn_val_scores, eval_results_val = evaluate_agent(
    agent_dqn,
    env_dqn,
    num_evaluation_episodes_dqn,
    max_steps_per_evaluation_episode_dqn,
    split="val",
    use_random_windows=True
)

# Test evaluation
dqn_test_scores, eval_results_test = evaluate_agent(
    agent_dqn,
    env_dqn,
    num_evaluation_episodes_dqn,
    max_steps_per_evaluation_episode_dqn,
    split="test",
    use_random_windows=True
)






# 7. Create a figure and a set of subplots
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 15), sharex=True)

# 8. Plot Current Capacity and Estimated Total CPU Load
axes[0].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['current_capacity'], label='Current Capacity')
axes[0].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['demand_cpu'], label='CPU Demand')
axes[0].set_ylabel('Capacity / Demand')
axes[0].set_title('DQN Agent Evaluation Results - Capacity and CPU Demand')
axes[0].legend()
axes[0].grid(True)

# 9. Plot Utilization and Target Utilization
axes[1].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['utilization'], label='Utilization')
axes[1].axhline(y=env_dqn.target_utilization, color='g', linestyle='--', label='Target Utilization')
axes[1].axhline(y=env_dqn.sla_threshold, color='r', linestyle='--', label='SLA Threshold')
axes[1].set_ylabel('Utilization')
axes[1].legend()
axes[1].grid(True)

# 10. Plot Reward
axes[2].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['reward'], label='Reward')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Reward')
axes[2].legend()
axes[2].grid(True)

# 11. Adjust layout and display plots
plt.tight_layout()
plt.show()

Using device: cpu
Episode 0	Average Score: -6671.03
Episode 10	Average Score: -3013.48
Episode 20	Average Score: -462.40
Episode 30	Average Score: -523.27
Episode 40	Average Score: -428.63
Episode 50	Average Score: -417.25
Episode 60	Average Score: -422.64
Episode 70	Average Score: -421.69
Episode 80	Average Score: -431.19
Episode 90	Average Score: -429.57
Episode 100	Average Score: -425.17
Episode 110	Average Score: -430.60
Episode 120	Average Score: -434.32
Episode 130	Average Score: -435.24
Episode 140	Average Score: -426.40
Episode 150	Average Score: -426.33
Episode 160	Average Score: -402.68
Episode 170	Average Score: -372.39
Episode 180	Average Score: -372.85
Episode 190	Average Score: -349.12
Episode 200	Average Score: -365.32
Episode 210	Average Score: -339.16
Episode 220	Average Score: -336.08
Episode 230	Average Score: -346.47
Episode 240	Average Score: -352.55
Episode 250	Average Score: -335.71
Episode 260	Average Score: -340.56
Episode 270	Average Score: -347.01
Episode 280

TypeError: reset_random_window() got an unexpected keyword argument 'split'

In [ ]:
def plot_evaluation_results(df, title_prefix, env):
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 15), sharex=True)

    # 1. Capacity & Demand
    axes[0].plot(df.index, df['current_capacity'], label='Capacity')
    axes[0].plot(df.index, df['demand_cpu'], label='CPU Demand')
    axes[0].set_ylabel("Capacity / Demand")
    axes[0].set_title(f"{title_prefix} - Capacity Tracking")
    axes[0].legend()
    axes[0].grid(True)

    # 2. Utilization
    axes[1].plot(df.index, df['utilization'], label='Utilization')
    axes[1].axhline(env.target_utilization, ls='--', c='g', label='Target Util')
    axes[1].axhline(env.sla_threshold, ls='--', c='r', label='SLA Threshold')
    axes[1].set_ylabel("Utilization")
    axes[1].set_title(f"{title_prefix} - Utilization")
    axes[1].legend()
    axes[1].grid(True)

    # 3. Reward
    axes[2].plot(df.index, df['reward'], label='Reward')
    axes[2].set_ylabel("Reward")
    axes[2].set_xlabel("Step")
    axes[2].set_title(f"{title_prefix} - Reward Timeline")
    axes[2].legend()
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_evaluation_results(train_df, "TRAIN SPLIT", env_dqn)
plot_evaluation_results(val_df, "VALIDATION SPLIT", env_dqn)
plot_evaluation_results(test_df, "TEST SPLIT", env_dqn)
